<a href="https://colab.research.google.com/github/OlhaZahrebelna/certflow-rag-assistant/blob/main/src/rag/03_end_to_end_rag_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers faiss-cpu openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 64.8 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
import faiss

from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI

In [3]:
from pathlib import Path

repo_path = Path("/content/certflow-rag-assistant")

if not repo_path.exists():
    !git clone https://github.com/OlhaZahrebelna/certflow-rag-assistant.git
else:
    print("Repository already exists.")

Cloning into 'certflow-rag-assistant'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 184 (delta 52), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (184/184), 294.73 KiB | 2.81 MiB/s, done.
Resolving deltas: 100% (52/52), done.


In [4]:
%cd /content/certflow-rag-assistant

/content/certflow-rag-assistant


In [5]:
chunks_path = Path("data/raw/processed/chunks.json")

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [chunk["content"] for chunk in chunks]

print(f"Loaded chunks: {len(chunks)}")

Loaded chunks: 81


In [6]:
embedding_model = SentenceTransformer(
    "multi-qa-MiniLM-L6-cos-v1"
)

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.asarray(
    embeddings,
    dtype="float32"
)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"Vectors indexed: {index.ntotal}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Vectors indexed: 81


In [7]:
def retrieve(query, k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    scores, indices = index.search(
        query_embedding,
        k=k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "chunk_id": chunk["chunk_id"],
            "document": chunk["metadata"]["title"],
            "section": chunk["metadata"]["section"],
            "content": chunk["content"],
        })

    return results

In [8]:
def build_context(results):
    context_parts = []

    for result in results:
        context_parts.append(
            f"""
Source {result['rank']}
Document: {result['document']}
Section: {result['section']}

{result['content']}
""".strip()
        )

    return "\n\n---\n\n".join(context_parts)

In [10]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

In [11]:
from openai import OpenAI

client = OpenAI(
    api_key=OPENAI_API_KEY
)

In [12]:
SYSTEM_PROMPT = """
You are CertFlow, an internal Account Data Certification assistant.

Answer the user's question using only the provided context.

Rules:
- Do not use outside knowledge.
- Do not invent policies or procedures.
- If the context does not contain enough information, say that the available documentation is insufficient.
- Keep the answer concise and operational.
- Cite the supporting document and section.
"""

In [13]:
def generate_answer(query, k=5):
    retrieved_chunks = retrieve(query, k=k)

    context = build_context(retrieved_chunks)

    user_prompt = f"""
Question:
{query}

Context:
{context}

Answer the question based only on the context above.
At the end, include a short Sources section with the document and section names you used.
"""

    response = client.responses.create(
        model="gpt-5-mini",
        instructions=SYSTEM_PROMPT,
        input=user_prompt
    )

    return {
        "query": query,
        "answer": response.output_text,
        "retrieved_chunks": retrieved_chunks
    }

In [14]:
result = generate_answer(
    "What evidence is required before an account can be certified?",
    k=5
)

print(result["answer"])

Required evidence and prerequisites before setting an account to Verified

- All mandatory fields in-scope are resolved (no open mandatory data items). (End-to-End Account Certification Workflow, §9)
- Supporting evidence is stored and traceable according to the Evidence Recording standard; for each certification record the analyst must capture:
  - source level (Primary or Secondary);
  - exact source/vendor name;
  - URL or internal source reference;
  - access or publication date;
  - which fields the source supports;
  - a relevant evidence excerpt in paraphrased form; and
  - the analyst’s conclusion.
  (Source Hierarchy and Evidence Standard, §6)
- Any changes to the account have documented reasons. (End-to-End Account Certification Workflow, §9)
- Duplicate screening is complete. (End-to-End Account Certification Workflow, §9)
- Required QA has passed. (End-to-End Account Certification Workflow, §9)
- If the intake request lacks a usable Account ID, the intake must include enoug

In [15]:
for item in result["retrieved_chunks"]:
    print(
        item["rank"],
        round(item["score"], 4),
        item["document"],
        "→",
        item["section"]
    )

1 0.5982 Account Data Certification Overview → 1. Purpose
2 0.5938 End-to-End Account Certification Workflow → 1. Intake
3 0.593 Account Certification Frequently Asked Questions → General
4 0.5573 End-to-End Account Certification Workflow → 9. Certification decision
5 0.5422 Source Hierarchy and Evidence Standard → 6. Evidence recording


## Baseline Result

The end-to-end RAG baseline successfully connects dense retrieval with LLM-based grounded answer generation.

The current pipeline uses:

- `multi-qa-MiniLM-L6-cos-v1` for embeddings;
- FAISS for dense retrieval;
- Top-5 retrieved chunks as context;
- an LLM for grounded answer generation with source attribution.

Initial testing showed that the generator can produce useful grounded answers, but some of the most relevant chunks are ranked below less relevant context.

This motivates the next experiment: adding a reranking stage between retrieval and generation.